In [ ]:
%env PYTORCH_ENABLE_MPS_FALLBACK=1
%env CUDA_VISIBLE_DEVICES=1
%cd ..

In [2]:
from typing import Iterable
import json
from pathlib import Path
from copy import deepcopy
from itertools import product
from collections import defaultdict

import numpy as np
import cv2

from matplotlib import pyplot as plt, colormaps as cm

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.io import read_image
from torchvision import transforms
from torchvision.transforms import functional as VF, InterpolationMode
from torchvision.models import googlenet, GoogLeNet_Weights

from tqdm import tqdm

images_dir = Path("data/dataset_crop")
labels_path = Path("data/result.json")

In [3]:
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

In [4]:
with open(labels_path, "r") as f:
    seg_labels = json.load(f)

for i in range(len(seg_labels["images"])):
    name = seg_labels["images"][i]["file_name"]
    name = images_dir / name.split("__")[-1]
    seg_labels["images"][i]["file_name"] = name

In [5]:
class ClassificationDataset(Dataset):
    def __init__(self, labels: dict, num_empty_samples: int, crop_size: int = 64):
        self.labels = labels
        self.crop_size = crop_size
        self.num_empty_samples = num_empty_samples

        self.category_counts = defaultdict(lambda: 0)
        self.category_counts[2] = num_empty_samples

        offsets = [crop_size // 2] * 2
        if crop_size % 2 == 1:
            offsets[1] += 1
        self.data = []

        unannotated_images = [True] * len(labels["images"])

        for annot in labels["annotations"]:
            img_id = annot["image_id"]
            unannotated_images[img_id] = False

            img_path = labels["images"][img_id]["file_name"]

            x_min, y_min, w, h = annot["bbox"]
            c1, c2 = int(x_min + w / 2), int(y_min + h / 2)

            bbox = [
                max(c2 - offsets[0], 0),
                max(c1 - offsets[1], 0),
                c2 + offsets[0],
                c1 + offsets[1],
            ]

            category = int(annot["category_id"] == 0)
            self.data.append((img_path, bbox, category))

            self.category_counts[category] += 1

        self.empty_images = [
            labels["images"][i]["file_name"]
            for i in range(len(unannotated_images))
            if unannotated_images[i]
        ]

    def _get_empty_sample(self):
        img_path = np.random.choice(self.empty_images)
        img = read_image(img_path)

        y = np.random.randint(0, img.shape[1] - self.crop_size + 1)
        x = np.random.randint(0, img.shape[2] - self.crop_size + 1)

        return img[:, y : y + self.crop_size, x : x + self.crop_size] / 255, 2

    def __len__(self):
        return len(self.data) + self.num_empty_samples

    def __getitem__(self, idx: int):
        if idx >= len(self.data):
            return self._get_empty_sample()

        img_path, bbox, category = self.data[idx]
        img = read_image(img_path)

        bbox[2] = min(bbox[2], img.shape[1])
        bbox[3] = min(bbox[3], img.shape[2])

        crop = img[:, bbox[0] : bbox[2], bbox[1] : bbox[3]] / 255

        h, w = crop.shape[1:]
        if h != self.crop_size or w != self.crop_size:
            dh = self.crop_size - h
            dw = self.crop_size - w
            to_pad = (dw // 2, dw // 2 + dw % 2, dh // 2, dh // 2 + dh % 2)
            crop = F.pad(crop, to_pad)

        return crop, category

In [6]:
augment_transform = transforms.Compose(
    [
        transforms.RandomAffine(
            degrees=45, translate=(0.2, 0.2), scale=(0.8, 1.2), shear=0.1
        ),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
    ]
)

def transform_collate_fn(batch: list[tuple[torch.Tensor, int]]):
    imgs, labels = zip(*batch)
    return torch.stack([augment_transform(img) for img in imgs], dim=0), torch.tensor(labels)

In [7]:
dataset = ClassificationDataset(
    seg_labels, num_empty_samples=400
)

train_dataset, val_dataset = random_split(
    dataset,
    [0.75, 0.25],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=16, collate_fn=transform_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=16)

In [ ]:
model = googlenet(weights=GoogLeNet_Weights.IMAGENET1K_V1)

model.fc = nn.Linear(1024, 3)

In [ ]:
model.load_state_dict(torch.load("/home/ai_n_zag@lab.graphicon.ru/tmp/kolobok/checkpoints/spike_classifier_0.9574.pt"))

In [9]:
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, labels: torch.Tensor):
        num_classes = logits.shape[-1]
        labels_ohe = F.one_hot(labels, num_classes)
        probs = F.softmax(logits, dim=-1)

        return torch.mean(
            torch.sum(
                -labels_ohe * torch.log(probs) * (1 - probs) ** self.gamma, dim=-1
            )
        )

In [10]:
def train_fn(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: str,
    n_epochs: int = 5,
):
    model.to(device)
    best_val_metric = -torch.inf
    best_model = deepcopy(model).cpu()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

    loss_fn = FocalLoss()

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0
        train_loop = tqdm(train_loader, desc=f"[{epoch}/{n_epochs}] Train Epoch")

        for i, (x, y) in enumerate(train_loop):
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            preds_raw = model(x)
            loss = loss_fn(preds_raw, y)
            loss.backward()

            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

            train_loop.set_postfix(loss=running_loss / (i + 1))

        model.eval()
        indicators = []
        eval_loop = tqdm(val_loader, desc=f"[{epoch}/{n_epochs}] Evaluation Epoch")

        with torch.no_grad():
            for i, (x, y) in enumerate(eval_loop):
                x = x.to(device)
                y = y.to(device)

                preds_raw = model(x).squeeze(1)
                indicators.extend((preds_raw.argmax(dim=-1) == y).cpu().tolist())

                eval_loop.set_postfix(acc=np.mean(indicators), best_acc=best_val_metric)

        val_metric = np.mean(indicators)

        if val_metric > best_val_metric:
            best_val_metric = val_metric
            best_model = deepcopy(model).cpu()

    return best_model, val_metric


In [11]:
def get_num_params(model: nn.Module):
    return sum(param.numel() for param in model.parameters())

print(f"number of model parameters: {get_num_params(model)}")

number of model parameters: 5602979


In [12]:
model, val_metric = train_fn(
    model,
    train_loader,
    val_loader,
    get_device(),
    n_epochs=25,
)
model.cpu()
torch.save(model.state_dict(), f"checkpoints/spike_classifier_{val_metric:.4f}.pt")

[25/25] Evaluation Epoch: 100%|██████████| 9/9 [00:04<00:00,  2.15it/s, acc=0.957, best_acc=0.959]


In [13]:
model.load_state_dict(torch.load(f"checkpoints/spike_classifier_{val_metric:.4f}.pt", weights_only=True))
# model.load_state_dict(torch.load("checkpoints/spike_classifier_0.0000.pt", weights_only=True))

<All keys matched successfully>